# SPY Exploratory Data Analysis

**Stage08 — SPY Next-Day High-Volatility Risk Alert**

This notebook reads the retained Stage07 snapshot, profiles its structure and derived EDA fields, and saves descriptive plots. It is a reference for future feature hypotheses, not a predictive or causal analysis.


## 1. Reproducible setup and lineage


In [1]:
from pathlib import Path
import os
import sys

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.ticker import PercentFormatter
import pandas as pd
import seaborn as sns
from IPython.display import display

if Path.cwd().name == "notebooks":
    os.chdir("..")
ROOT = Path.cwd()
if not (ROOT / "src" / "eda.py").is_file():
    for candidate in (ROOT, *ROOT.parents):
        project_candidate = candidate / "project"
        if (project_candidate / "src" / "eda.py").is_file():
            ROOT = project_candidate
            break
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.eda import eda_summary, prepare_spy_eda_frame
from src.storage import read_df

PROCESSED_DIR = ROOT / "data" / "processed"
REPORTS_DIR = ROOT / "reports"
REPORTS_DIR.mkdir(parents=True, exist_ok=True)
source_paths = sorted(PROCESSED_DIR.glob("spy_return_outlier_flags_*.parquet"))
if not source_paths:
    raise FileNotFoundError("Run the Stage07 pipeline before this EDA notebook.")
source_path = source_paths[-1]
snapshot_timestamp = source_path.stem.removeprefix("spy_return_outlier_flags_")
print("Project root:", ROOT)
print("Stage07 source:", source_path.name)


Project root: /Users/cengchengyu/Documents/NYU/Class/Boot Camp 4/CS HW/bootcamp_chengyu_zeng/project
Stage07 source: spy_return_outlier_flags_20260907-143336.parquet


## 2. Structure, integrity, and reusable summary


In [2]:
stage07_frame = read_df(source_path)
spy_eda = prepare_spy_eda_frame(stage07_frame)
summaries = eda_summary(spy_eda)

print("Base shape:", stage07_frame.shape)
print("EDA shape:", spy_eda.shape)
display(summaries["overview"])
display(summaries["column_profile"])
display(summaries["numeric_summary"])
print("Attention before feature engineering")
display(summaries["attention"])


Base shape: (2512, 9)
EDA shape: (2512, 16)


,rows,columns,missing_cells,duplicate_rows,memory_bytes
0,2512,16,23,0,251630


,column,role,dtype,non_null,missing_count,missing_fraction,unique_count,dominant_fraction,near_zero_variance,dominant_category,attention
0,date,datetime,datetime64[us],2512,0,0.000000,2512,0.000398,False,False,none
1,open,numeric,float64,2512,0,0.000000,2432,0.001194,False,False,none
2,high,numeric,float64,2512,0,0.000000,2432,0.001194,False,False,none
3,low,numeric,float64,2512,0,0.000000,2438,0.000796,False,False,none
4,close,numeric,float64,2512,0,0.000000,2436,0.000796,False,False,none
5,volume,numeric,int64,2512,0,0.000000,2511,0.000796,False,False,none
6,daily_return,numeric,float64,2511,1,0.000398,2507,0.001990,False,False,has_missing
7,return_outlier_iqr,categorical,bool,2512,0,0.000000,2,0.934315,False,False,none
8,return_outlier_zscore,categorical,bool,2512,0,0.000000,2,0.986863,False,True,dominant_category
9,abs_return,numeric,float64,2511,1,0.000398,2507,0.001990,False,False,has_missing


,column,count,mean,std,min,25%,50%,75%,max,missing_count,missing_fraction,skew,kurtosis,unique_count
0,open,2512.0,4.108343e+02,1.460820e+02,2.089100e+02,2.824125e+02,3.944100e+02,5.024875e+02,7.785400e+02,0,0.000000,0.679790,-0.509267,2432
1,high,2512.0,4.130406e+02,1.466843e+02,2.098900e+02,2.840962e+02,3.971400e+02,5.065888e+02,7.793700e+02,0,0.000000,0.674487,-0.515647,2432
2,low,2512.0,4.084289e+02,1.453950e+02,2.083800e+02,2.806572e+02,3.916900e+02,4.995371e+02,7.754301e+02,0,0.000000,0.684851,-0.502440,2438
3,close,2512.0,4.109036e+02,1.461121e+02,2.085500e+02,2.824600e+02,3.949300e+02,5.035000e+02,7.778800e+02,0,0.000000,0.679234,-0.510450,2436
4,volume,2512.0,7.920013e+07,3.854229e+07,9.999999e+06,5.589882e+07,7.086976e+07,9.147290e+07,3.922207e+08,0,0.000000,2.687621,12.116723,2511
5,daily_return,2511.0,5.656502e-04,1.136301e-02,-1.094237e-01,-3.706364e-03,6.988516e-04,5.991378e-03,1.050193e-01,1,0.000398,-0.310705,14.645793,2507
6,abs_return,2511.0,7.384577e-03,8.653580e-03,0.000000e+00,2.057864e-03,5.052296e-03,9.707781e-03,1.094237e-01,1,0.000398,4.093180,30.422074,2507
7,intraday_range,2512.0,1.137062e-02,8.961650e-03,1.229256e-03,5.946043e-03,8.852325e-03,1.369107e-02,1.126175e-01,0,0.000000,3.387445,20.208032,2512
8,close_to_open_return,2512.0,1.945330e-04,8.563322e-03,-5.661155e-02,-3.338779e-03,5.453690e-04,4.279441e-03,1.118272e-01,0,0.000000,0.551315,15.405778,2505
9,log_volume,2512.0,1.809917e+01,4.045979e-01,1.611810e+01,1.783905e+01,1.807635e+01,1.833155e+01,1.978734e+01,0,0.000000,0.477422,1.064983,2511


Attention before feature engineering


,column,role,missing_count,missing_fraction,dominant_fraction,attention
0,daily_return,numeric,1,0.000398,0.001990,has_missing
1,return_outlier_zscore,categorical,0,0.000000,0.986863,dominant_category
2,abs_return,numeric,1,0.000398,0.001990,has_missing
3,rolling_volatility_21,numeric,21,0.008360,0.008360,has_missing


## 3. Distributions

**What:** return tails, range, and volume shape. **So what:** normality is a weak assumption. **Now what:** preserve tail sessions and test robust, leakage-safe future features.


In [3]:
distribution_path = REPORTS_DIR / f"spy_eda_distributions_{snapshot_timestamp}.png"
fig, axes = plt.subplots(2, 2, figsize=(12, 8.5))
sns.histplot(spy_eda["daily_return"].dropna(), bins=80, kde=True, color="#2563EB", ax=axes[0, 0])
axes[0, 0].set_title("Daily Close-to-Close Return")
axes[0, 0].xaxis.set_major_formatter(PercentFormatter(1.0))
sns.boxplot(x=spy_eda["abs_return"], color="#93C5FD", ax=axes[0, 1])
axes[0, 1].set_title("Absolute Return and Tail Observations")
axes[0, 1].xaxis.set_major_formatter(PercentFormatter(1.0))
sns.histplot(spy_eda["intraday_range"], bins=80, color="#10B981", ax=axes[1, 0])
axes[1, 0].set_title("Intraday Range")
axes[1, 0].xaxis.set_major_formatter(PercentFormatter(1.0))
sns.histplot(spy_eda["log_volume"], bins=80, color="#F59E0B", ax=axes[1, 1])
axes[1, 1].set_title("Log Volume")
fig.suptitle("SPY Stage08 Distributional EDA", y=1.01)
fig.tight_layout()
fig.savefig(distribution_path, dpi=170, bbox_inches="tight")
plt.close(fig)
print("Saved:", distribution_path.name)


Saved: spy_eda_distributions_20260907-143336.png


## 4. Relationships and correlation

These are contemporaneous associations, not next-day predictions. Stage09 must lag any candidate predictor before evaluating it.


In [4]:
relationship_path = REPORTS_DIR / f"spy_eda_relationships_{snapshot_timestamp}.png"
relationship_data = spy_eda.dropna(subset=["abs_return", "intraday_range", "log_volume"])
correlation_columns = ["daily_return", "abs_return", "intraday_range", "close_to_open_return", "log_volume", "rolling_volatility_21"]
correlation = spy_eda[correlation_columns].corr()
fig, axes = plt.subplots(1, 3, figsize=(17, 5))
sns.scatterplot(data=relationship_data, x="intraday_range", y="abs_return", hue="return_outlier_iqr", palette={False: "#2563EB", True: "#DC2626"}, alpha=0.45, s=22, linewidth=0, ax=axes[0])
axes[0].set_title("Absolute Return vs Intraday Range")
axes[0].xaxis.set_major_formatter(PercentFormatter(1.0))
axes[0].yaxis.set_major_formatter(PercentFormatter(1.0))
sns.scatterplot(data=relationship_data, x="log_volume", y="abs_return", hue="return_outlier_iqr", palette={False: "#2563EB", True: "#DC2626"}, alpha=0.45, s=22, linewidth=0, legend=False, ax=axes[1])
axes[1].set_title("Absolute Return vs Log Volume")
axes[1].yaxis.set_major_formatter(PercentFormatter(1.0))
sns.heatmap(correlation, annot=True, fmt=".2f", cmap="vlag", vmin=-1, vmax=1, center=0, square=True, ax=axes[2])
axes[2].set_title("Contemporaneous Correlation")
fig.tight_layout()
fig.savefig(relationship_path, dpi=170, bbox_inches="tight")
plt.close(fig)
print("Saved:", relationship_path.name)
display(correlation)


Saved: spy_eda_relationships_20260907-143336.png


,daily_return,abs_return,intraday_range,close_to_open_return,log_volume,rolling_volatility_21
daily_return,1.000000,-0.055777,-0.110967,0.766692,-0.176423,0.024989
abs_return,-0.055777,1.000000,0.718589,-0.000406,0.539158,0.530175
intraday_range,-0.110967,0.718589,1.000000,-0.028928,0.684293,0.645059
close_to_open_return,0.766692,-0.000406,-0.028928,1.000000,-0.117053,0.050288
log_volume,-0.176423,0.539158,0.684293,-0.117053,1.000000,0.412156
rolling_volatility_21,0.024989,0.530175,0.645059,0.050288,0.412156,1.000000


## 5. Time-series structure

A rising price level and clustered rolling volatility imply that raw price levels are not stationary risk predictors and random train/test splits would mix regimes.


In [5]:
time_series_path = REPORTS_DIR / f"spy_eda_time_series_{snapshot_timestamp}.png"
fig, axes = plt.subplots(2, 1, figsize=(13, 8), sharex=True)
axes[0].plot(spy_eda["date"], spy_eda["close"], color="#2563EB", linewidth=1.2)
axes[0].set_title("SPY Unadjusted Closing Price")
axes[0].set_ylabel("Price")
axes[1].plot(spy_eda["date"], spy_eda["rolling_volatility_21"], color="#DC2626", linewidth=1.0)
axes[1].axhline(spy_eda["rolling_volatility_21"].median(), color="#374151", linestyle="--", linewidth=0.9, label="median")
axes[1].set_title("21-Session Annualized Rolling Volatility")
axes[1].set_ylabel("Annualized volatility")
axes[1].legend()
fig.tight_layout()
fig.savefig(time_series_path, dpi=170, bbox_inches="tight")
plt.close(fig)
print("Saved:", time_series_path.name)


Saved: spy_eda_time_series_20260907-143336.png


## 6. Findings, assumptions, and implications

1. **Tails and skew matter.** Extreme returns and right-skewed range/volume measures motivate robust, tail-aware feature and evaluation choices rather than automatic deletion.
2. **Risk measures move together but are not interchangeable.** Range, absolute return, volume, and rolling volatility are useful feature candidates, but correlation is not forecasting evidence and may reveal redundancy.
3. **Risk clusters over time.** Use chronological splits and compute all lag/rolling features using only information available by each close.

The source is unadjusted OHLCV. Structural warm-up missingness in return and rolling volatility should be handled only after the future target and feature alignment are defined.


In [6]:
required_reports = [distribution_path, relationship_path, time_series_path]
assert len(spy_eda) == len(stage07_frame)
assert spy_eda["date"].is_unique and spy_eda["date"].is_monotonic_increasing
assert int(spy_eda["daily_return"].isna().sum()) == 1
assert int(spy_eda["rolling_volatility_21"].isna().sum()) == 21
assert all(path.exists() and path.stat().st_size > 0 for path in required_reports)
print("Stage08 EDA notebook checks passed.")


Stage08 EDA notebook checks passed.
